# ShadowNet Training (Colab / Kaggle / HF Space)

End-to-end fine-tuning with **optional Weights & Biases** logging. Works on Google Colab, Kaggle, and Hugging Face Spaces (JupyterLab). This notebook is the main runnable training artifact.

**Base model:** `Qwen/Qwen2.5-1.5B-Instruct`  
**Method:** supervised fine-tuning with LoRA-friendly training flow  
**Recommended runtime:** T4 GPU or better

**Repo:** [shadownet-defence](https://github.com/salim7-s/shadownet-defence)  
**HF Space:** [zizoha/shadownet-Cops](https://huggingface.co/spaces/zizoha/shadownet-Cops)

## 0. Before You Start

In Colab, set:
- `Runtime -> Change runtime type -> GPU`
- choose **T4** or better

What this notebook produces:
- generated SFT dataset JSONL files
- trained adapter saved to `shadownet-sft`
- loss graph at `training/sft_loss_curve.png`
- optional trained evaluation in `artifacts/`

Recommended success checks:
1. Install completes.
2. Dataset files appear in `training/`.
3. Training saves to `shadownet-sft`.
4. `training/sft_loss_curve.png` is created.
5. Optional: `artifacts/eval_summary.json` and `artifacts/eval_summary.md` are created.

In [ ]:
import os, sys

REPO = "shadownet-defence"

# Clone only if we are not already inside the repo.
# On HF Spaces / Kaggle the working directory may already be the repo root.
if not os.path.exists("environment.py"):
    os.system("git clone https://github.com/salim7-s/" + REPO + ".git")
    os.chdir(REPO)
    sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())

!pip install -q -r requirements.txt
!pip install -q -U "trl>=0.12" "transformers>=4.45" accelerate peft torch datasets wandb
!python scripts/verify_core.py

In [ ]:
import os
import wandb

os.environ["WANDB_PROJECT"] = os.environ.get("WANDB_PROJECT", "shadownet-sft")

# Try Colab Secrets first; fall back to env var (works on HF Spaces / Kaggle).
_wandb_key = os.environ.get("WANDB_API_KEY")
try:
    from google.colab import userdata as _ud
    _wandb_key = _wandb_key or _ud.get("WANDB_API_KEY")
except Exception:
    pass  # Not running in Colab

if _wandb_key:
    wandb.login(key=_wandb_key)
    print("W&B login successful.")
else:
    print("No WANDB_API_KEY found. Continuing without W&B (set report_to='none' below if needed).")

## 2. Optional W&B Setup

If you want a public training run URL, add a Colab secret named `WANDB_API_KEY`.

If you skip this step, training can still run locally. You can also change `report_to` later to `"none"`.

In [ ]:
import os
import wandb

os.environ["WANDB_PROJECT"] = os.environ.get("WANDB_PROJECT", "shadownet-sft")

try:
    from google.colab import userdata
    key = userdata.get("WANDB_API_KEY")
    if key:
        wandb.login(key=key)
        print("W&B login successful.")
    else:
        print("No WANDB_API_KEY secret found. Continuing without W&B login.")
except Exception:
    print("Colab Secrets unavailable. Continuing without W&B login.")

## 3. Build SFT Data (JSONL)

The dataset is generated from expert-like baseline rollouts across all scenarios and attacker profiles.

- `training/sft_6x.jsonl`: all completed episodes, repeated with multiple seeds
- `training/sft_high_6x.jsonl`: keeps only higher-quality episodes using a score filter

In [ ]:
!python training/build_sft_dataset.py -o training/sft_6x.jsonl --repeats 6
!python training/build_sft_dataset.py -o training/sft_high_6x.jsonl --min-score 0.55 --repeats 6

## 4. Preview One Training Example

This is a quick sanity check that the generated dataset is real and formatted correctly.

In [ ]:
import json
from pathlib import Path

dataset_path = Path("training/sft_high_6x.jsonl")
with open(dataset_path, "r", encoding="utf-8") as f:
    sample = json.loads(next(f))

print("Dataset:", dataset_path)
print("Messages:", len(sample["messages"]))
print()
for msg in sample["messages"]:
    print(msg["role"].upper())
    print(msg["content"][:700])
    print("\n" + "-" * 80 + "\n")

## 5. Fine-Tune with TRL

This notebook uses `SFTTrainer` with a `formatting_func` because the dataset is stored as chat-style `messages`.

If you later switch to Unsloth, keep the same formatting function.

In [ ]:
import os

import wandb
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
output_dir = "shadownet-sft"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto", device_map="auto")

# Prefer the filtered dataset first; switch to training/sft_6x.jsonl if you want more volume.
ds = load_dataset("json", data_files="training/sft_high_6x.jsonl", split="train")

def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

report_to = "wandb"

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    processing_class=tokenizer,
    formatting_func=formatting_func,
    args=SFTConfig(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.10,
        logging_steps=1,
        save_strategy="epoch",
        report_to=report_to,
        run_name="qwen2.5-1.5b-shadownet-sft-high6x",
    ),
)

train_result = trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

try:
    wandb.finish()
except Exception:
    pass

print("Training complete.")
print("Output directory:", output_dir)
print("Reported training loss:", getattr(train_result, "training_loss", "n/a"))

## 6. Save a Spike-Style Loss Plot

This creates a screenshot-friendly training figure from `trainer_state.json`.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

state_file = Path("shadownet-sft") / "trainer_state.json"
if not state_file.exists():
    checkpoints = sorted(Path("shadownet-sft").glob("checkpoint-*"))
    if not checkpoints:
        raise FileNotFoundError("No trainer_state.json or checkpoint-* directory found in shadownet-sft")
    state_file = checkpoints[-1] / "trainer_state.json"

with open(state_file, "r", encoding="utf-8") as f:
    state = json.load(f)

log_history = state["log_history"]
steps = [x["step"] for x in log_history if "loss" in x]
losses = [x["loss"] for x in log_history if "loss" in x]
window = max(3, min(9, len(losses)))
smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
smooth_steps = steps[window - 1:]

plt.figure(figsize=(11, 4.5))
baseline = min(losses) - 0.02
plt.vlines(steps, baseline, losses, color="#2E86AB", alpha=0.18, linewidth=1.0, label="raw loss spikes")
plt.scatter(steps, losses, s=16, color="#2E86AB", alpha=0.55)
plt.plot(smooth_steps, smoothed, linewidth=3, color="#F18F01", label=f"{window}-step moving avg")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title("ShadowNet SFT Loss Curve (Spike View)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("training/sft_loss_curve.png", dpi=150)
plt.show()

print(f"Saved training/sft_loss_curve.png from {state_file}")
print(f"First logged loss: {losses[0]:.4f}")
print(f"Last logged loss:  {losses[-1]:.4f}")
print(f"Logged steps:      {len(steps)}")

## 7. Baseline Evaluation (Optional, CPU-Friendly)

This regenerates the baseline comparison files used elsewhere in the repo.

In [ ]:
!python scripts/eval_baseline.py --seed 42

## 8. Evaluate the Trained Checkpoint (Optional, GPU)

This tests the trained policy inside the live ShadowNet environment and writes summaries into `artifacts/`.

In [ ]:
!python scripts/eval_trained.py \
  --checkpoint shadownet-sft \
  --base-model Qwen/Qwen2.5-1.5B-Instruct \
  --seed 42 \
  --episodes 5

## 9. Optional RL / GRPO Follow-On

`training/reward_adapter.py` exposes `EpisodeRunner` and reward shaping utilities if you want to continue from SFT into RL later.

In [ ]:
from pathlib import Path

print("Files currently available:")
for path in [
    Path("training/sft_loss_curve.png"),
    Path("training/eval_baseline.json"),
    Path("training/eval_baseline_table.md"),
    Path("artifacts/eval_summary.json"),
    Path("artifacts/eval_summary.md"),
    Path("shadownet-sft"),
]:
    print(f"- {path}: {'FOUND' if path.exists() else 'missing'}")

## 10. Optional Download / Export

Use one of these patterns if you want to save the adapter or graph outside the Colab runtime.

In [ ]:
# Example: zip the trained checkpoint
# !zip -r shadownet-sft.zip shadownet-sft

# Example: download files locally
# from google.colab import files
# files.download('shadownet-sft.zip')
# files.download('training/sft_loss_curve.png')

# Example: copy to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r shadownet-sft /content/drive/MyDrive/
# !cp training/sft_loss_curve.png /content/drive/MyDrive/